## CXRFE (Chest X-ray Fact Encoder)

In [1]:
from medvqa.models.huggingface_utils import _adapt_checkpoint_keys
from medvqa.models.checkpoint import load_model_state_dict, get_checkpoint_filepath
from medvqa.utils.logging_utils import setup_logging
from transformers import AutoTokenizer, AutoModel
import torch
setup_logging()

2025-09-18 16:14:27,299 - INFO - root - Logging configured (Color: True).


In [2]:
# model_checkpoint_folder_path = '/mnt/data/pamessina/workspaces/medvqa-workspace/models/fact_embedding/20240629_084405_MIMIC-CXR(triplets+classif+entcont+nli+radgraph+autoencoder)_FactEncoder(microsoft-BiomedVLP-CXR-BERT-specialized)'
model_checkpoint_folder_path = '/mnt/data/pamessina/workspaces/medvqa-workspace/models/fact_embedding/20250610_213953_MIMIC-CXR(triplets+classif+entcont+nli+autoencoder)_FactEncoder(microsoft-BiomedVLP-CXR-BERT-specialized)'
# model = AutoModel.from_pretrained('microsoft/BiomedVLP-CXR-BERT-specialized', revision="6cfc310817fb7d86762d888ced1e3709c57ac578", trust_remote_code=True)
model = AutoModel.from_pretrained('microsoft/BiomedVLP-CXR-BERT-specialized', trust_remote_code=True)
model_checkpoint_filepath = get_checkpoint_filepath(model_checkpoint_folder_path)
checkpoint = torch.load(model_checkpoint_filepath)
load_model_state_dict(model, _adapt_checkpoint_keys(checkpoint['model']), strict=False)

2025-09-18 16:14:30,399 - INFO - medvqa.models.checkpoint - checkpoint_names = ['checkpoint_154_cacc+chf1+chf1+cscc+encc+hscc+nlcc+sass+ta0)+ta1)+ta2)+ta0)+ta1)+ta2)+ta3)+ta4)+ta5)+ta6)+ta7)+ta8)=0.9070.pt']
2025-09-18 16:14:31,249 - WARNING - medvqa.models.checkpoint - model state dict has 210 keys, loaded state dict has 252 keys, intersection has 210 keys, union has 252 keys.
Examples of keys in loaded state dict but not in model:
  fact_decoder.decoder.layers.0.norm1.weight
  fact_decoder.decoder.layers.0.norm1.bias
  fact_decoder.decoder.layers.0.self_attn.in_proj_bias
  nli_hidden_layer.weight
  fact_decoder.decoder.layers.0.norm2.bias
  chest_imagenome_anatloc_classifier.weight
  chest_imagenome_obs_classifier.weight
  fact_decoder.decoder.layers.0.norm2.weight
  nli_classifier.weight
  fact_decoder_embedding_table.weight


In [3]:
model.config

CXRBertConfig {
  "_name_or_path": "microsoft/BiomedVLP-CXR-BERT-specialized",
  "architectures": [
    "CXRBertModel"
  ],
  "attention_probs_dropout_prob": 0.25,
  "auto_map": {
    "AutoConfig": "microsoft/BiomedVLP-CXR-BERT-specialized--configuration_cxrbert.CXRBertConfig",
    "AutoModel": "microsoft/BiomedVLP-CXR-BERT-specialized--modeling_cxrbert.CXRBertModel"
  },
  "classifier_dropout": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.25,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "cxr-bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "projection_size": 128,
  "torch_dtype": "float32",
  "transformers_version": "4.41.2",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}

In [5]:
# model.save_pretrained("/home/pamessina/huggingface_models/CXRFE_debug/", revision="refs/pr/5")
model.save_pretrained("/home/pamessina/huggingface_models/CXRFE/")

In [6]:
# tokenizer = AutoTokenizer.from_pretrained('microsoft/BiomedVLP-CXR-BERT-specialized', revision="refs/pr/5", trust_remote_code=True)
# tokenizer.save_pretrained("/home/pamessina/huggingface_models/CXRFE_test/")
tokenizer = AutoTokenizer.from_pretrained('microsoft/BiomedVLP-CXR-BERT-specialized', trust_remote_code=True)
tokenizer.save_pretrained("/home/pamessina/huggingface_models/CXRFE/")

('/home/pamessina/huggingface_models/CXRFE/tokenizer_config.json',
 '/home/pamessina/huggingface_models/CXRFE/special_tokens_map.json',
 '/home/pamessina/huggingface_models/CXRFE/vocab.txt',
 '/home/pamessina/huggingface_models/CXRFE/added_tokens.json')

In [7]:
!ls -lh /home/pamessina/huggingface_models/CXRFE/

total 419M
-rw-rw-r-- 1 pamessina pamessina  927 Sep 18 16:15 config.json
-rw-rw-r-- 1 pamessina pamessina 1.1K Jul  1  2024 configuration_cxrbert.py
-rw-rw-r-- 1 pamessina pamessina   90 Sep 18 16:15 generation_config.json
-rw-rw-r-- 1 pamessina pamessina 6.1K Jul  1  2024 modeling_cxrbert.py
-rw-rw-r-- 1 pamessina pamessina 419M Sep 18 16:15 model.safetensors
-rw-rw-r-- 1 pamessina pamessina   31 Jun 30  2024 README.md
-rw-rw-r-- 1 pamessina pamessina  125 Sep 18 16:16 special_tokens_map.json
-rw-rw-r-- 1 pamessina pamessina 1.4K Sep 18 16:16 tokenizer_config.json
-rw-rw-r-- 1 pamessina pamessina 230K Sep 18 16:16 vocab.txt


## T5 Fact Extractor

In [9]:
from medvqa.utils.files_utils import list_filepaths_with_prefix_and_timestamps

In [11]:
list_filepaths_with_prefix_and_timestamps('/mnt/data/pamessina/workspaces/medvqa-workspace/models/seq2seq/2025',
                                          must_contain=['checkpoint'])

[('/mnt/data/pamessina/workspaces/medvqa-workspace/models/seq2seq/20250704_024421_sentence2facts(S2F)_Seq2Seq(t5-small)/checkpoint_163_s2s_loss=0.9210.pt',
  '2025-07-04 05:53:02'),
 ('/mnt/data/pamessina/workspaces/medvqa-workspace/models/seq2seq/20250703_222506_sentence2facts(S2F)_Seq2Seq(t5-small)/checkpoint_199_s2s_loss=0.9003.pt',
  '2025-07-04 02:24:42'),
 ('/mnt/data/pamessina/workspaces/medvqa-workspace/models/seq2seq/20250129_162100_sentence2facts(S2F)_Seq2Seq(t5-small)/checkpoint_48_s2s_loss=0.8960.pt',
  '2025-01-29 17:42:46'),
 ('/mnt/data/pamessina/workspaces/medvqa-workspace/models/seq2seq/20250129_110135_sentence2facts(S2F)_Seq2Seq(t5-small)/checkpoint_170_s2s_loss=0.8984.pt',
  '2025-01-29 16:15:34'),
 ('/mnt/data/pamessina/workspaces/medvqa-workspace/models/seq2seq/20250129_110135_sentence2facts(S2F)_Seq2Seq(t5-small)/checkpoint_128_s2s_loss=0.8939.pt',
  '2025-01-29 14:58:12'),
 ('/mnt/data/pamessina/workspaces/medvqa-workspace/models/seq2seq/20250129_093901_sentence2

In [8]:
from transformers import T5ForConditionalGeneration
from transformers import T5TokenizerFast

In [12]:
model = T5ForConditionalGeneration.from_pretrained('t5-small')
# model_checkpoint_folder_path = '/mnt/data/pamessina/workspaces/medvqa-workspace/models/seq2seq/20240320_195545_sentence2facts(S2F)_Seq2Seq(t5-small)/'
model_checkpoint_folder_path = '/mnt/data/pamessina/workspaces/medvqa-workspace/models/seq2seq/20250704_024421_sentence2facts(S2F)_Seq2Seq(t5-small)/'
model_checkpoint_filepath = get_checkpoint_filepath(model_checkpoint_folder_path)
checkpoint = torch.load(model_checkpoint_filepath)
load_model_state_dict(model, _adapt_checkpoint_keys(checkpoint['model']), strict=False)

2025-09-18 16:28:48,146 - INFO - medvqa.models.checkpoint - checkpoint_names = ['checkpoint_163_s2s_loss=0.9210.pt']


In [13]:
model.save_pretrained("/home/pamessina/huggingface_models/T5FactExtractor/")

In [14]:
tokenizer = T5TokenizerFast.from_pretrained('t5-small')
tokenizer.save_pretrained("/home/pamessina/huggingface_models/T5FactExtractor/")

('/home/pamessina/huggingface_models/T5FactExtractor/tokenizer_config.json',
 '/home/pamessina/huggingface_models/T5FactExtractor/special_tokens_map.json',
 '/home/pamessina/huggingface_models/T5FactExtractor/spiece.model',
 '/home/pamessina/huggingface_models/T5FactExtractor/added_tokens.json',
 '/home/pamessina/huggingface_models/T5FactExtractor/tokenizer.json')